# Embedder — multilingual-e5-large → NPY

Genera embeddings de todos los registros de `eventos.db` con el modelo  
`intfloat/multilingual-e5-large` y los guarda en disco (`.npy` + índice CSV).

**Flujo:**
1. Leer la BD → construir `texto_embedding` por registro  
2. Cargar el modelo `multilingual-e5-large`  
3. Vectorizar en batches con barra de progreso  
4. Guardar `embeddings.npy` + `embeddings_index.csv`  
5. Verificación rápida de integridad  


## 0 · Dependencias

In [1]:
# Descomenta si no tienes instaladas las librerías
# !pip install sentence-transformers numpy pandas tqdm


## 1 · Configuración

In [ ]:
from pathlib import Path

# 1. Definimos la ruta base: sube un nivel (..) a Desafio-Data, entra en data y luego en descargas
BASE_DIR = Path("..") / "data" / "descargas"

# 2. Aplicamos la base a tus variables
DB_PATH         = BASE_DIR / "eventos.db"          # eventos.db estará en data/descargas/

OUTPUT_DIR      = BASE_DIR / "embeddings"          # carpeta embeddings dentro de descargas/
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)      # Crea la ruta completa si no existe

EMB_PATH        = OUTPUT_DIR / "embeddings.npy"        
INDEX_PATH      = OUTPUT_DIR / "embeddings_index.csv"

# ── Modelo ──────────────────────────────────────────────────────────────
MODELO          = "intfloat/multilingual-e5-large"

# ── Parámetros de encoding ───────────────────────────────────────────────
BATCH_SIZE      = 64    # ajusta según tu VRAM/RAM
NORMALIZE       = True  # True → similitud coseno = producto escalar

print(f"DB          : {DB_PATH}")
print(f"Salida      : {OUTPUT_DIR}/")
print(f"Modelo      : {MODELO}")
print(f"Batch size  : {BATCH_SIZE}")


DB          : eventos.db
Salida      : npc/
Modelo      : intfloat/multilingual-e5-large
Batch size  : 64


## 2 · Cargar datos desde SQLite

In [3]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)
df   = pd.read_sql_query("SELECT * FROM eventos", conn)
conn.close()

print(f"Registros cargados : {len(df):,}")
print(f"Columnas           : {df.columns.tolist()}")
df.head(3)


Registros cargados : 4,070
Columnas           : ['id', 'business_id', 'fuente', 'external_id', 'title', 'description', 'categoria', 'tipo_plantilla', 'municipio', 'territorio', 'address', 'lat', 'lng', 'telefono', 'email', 'website', 'es_interior', 'es_carrito', 'es_cambiador', 'es_silla_ruedas', 'es_mascotas', 'edad_minima', 'fecha_inicio', 'fecha_fin', 'lugar', 'price', 'imagen_url', 'tipo_evento']


,id,business_id,fuente,external_id,title,description,categoria,tipo_plantilla,municipio,territorio,...,es_cambiador,es_silla_ruedas,es_mascotas,edad_minima,fecha_inicio,fecha_fin,lugar,price,imagen_url,tipo_evento
0,1,None,https://turismoa.euskadi.eus/es/centros-comerc...,None,Dendaraba,None,comercio,Centros comerciales,Vitoria-Gasteiz,araba,...,1,1,0,0,2026-06-02,None,None,0,None,None
1,2,None,https://turismoa.euskadi.eus/es/centros-comerc...,None,El Boulevard,None,comercio,Centros comerciales,Vitoria-Gasteiz,araba,...,1,1,0,0,2026-06-02,None,None,0,None,None
2,3,None,https://turismoa.euskadi.eus/es/centros-comerc...,None,Parque Comercial Gorbeia,None,comercio,Centros comerciales,Zigoitia,araba,...,1,1,0,0,2026-06-02,None,None,0,None,None


## 3 · Construir `texto_embedding`

El modelo multilingual-e5 requiere el prefijo `passage:` en los documentos  
y `query:` en las consultas. Aquí añadimos `passage:`.


In [4]:
def construir_texto(row) -> str:
    campos = [
        str(row.get("external_id",  "") or ""),
        str(row.get("categoria",    "") or ""),
        str(row.get("tipo_evento",  "") or ""),
        str(row.get("description",  "") or ""),
        str(row.get("municipio",    "") or ""),
    ]

    # Edad mínima
    edad = row.get("edad_minima")
    if edad is not None and str(edad).strip() not in ("", "0", "None"):
        campos.append(f"edad mínima {edad} años")

    # Booleanos → texto natural
    flags = []
    if row.get("es_interior"):     flags.append("interior")
    if row.get("es_carrito"):      flags.append("accesible con carrito")
    if row.get("es_cambiador"):    flags.append("con cambiador")
    if row.get("es_silla_ruedas"): flags.append("accesible silla de ruedas")
    if row.get("es_mascotas"):     flags.append("admite mascotas")
    if flags:
        campos.append(", ".join(flags))

    cuerpo = ". ".join(c.strip() for c in campos if c.strip())
    return f"passage: {cuerpo}"

df["texto_embedding"] = df.apply(construir_texto, axis=1)

# Descartar registros sin texto útil
df = df[df["texto_embedding"].str.len() > len("passage: ")].reset_index(drop=True)

print(f"Registros con texto : {len(df):,}")
print("\nEjemplos:")
for txt in df["texto_embedding"].head(3):
    print(f"  {txt[:120]}")


Registros con texto : 4,070

Ejemplos:
  passage: comercio. Vitoria-Gasteiz. interior, accesible con carrito, con cambiador, accesible silla de ruedas
  passage: comercio. Vitoria-Gasteiz. interior, accesible con carrito, con cambiador, accesible silla de ruedas
  passage: comercio. Zigoitia. interior, accesible con carrito, con cambiador, accesible silla de ruedas


In [5]:
print(df[["external_id", "description", "tipo_evento", "categoria", "municipio"]].sample(10, random_state=42).to_string())

     external_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   description             tipo_evento        categoria                 municipio
605         None  El restaurante La Revelia en su interior se diseña en tonos oscuros con paredes y revestimientos en 

## 4 · Cargar modelo

In [6]:
from sentence_transformers import SentenceTransformer
import time

print(f"Cargando {MODELO} ...")
t0    = time.perf_counter()
model = SentenceTransformer(MODELO)
t1    = time.perf_counter()
print(f"Modelo cargado en {t1-t0:.1f} s")

# Dimensión del vector (1024 para e5-large)
dim_test = model.encode("passage: test", normalize_embeddings=True)
print(f"Dimensión del vector : {len(dim_test)}")


c:\Users\rns_2\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando intfloat/multilingual-e5-large ...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2981.42it/s]


Modelo cargado en 8.8 s
Dimensión del vector : 1024


## 5 · Vectorizar en batches

Los embeddings se generan en batches para no saturar la memoria.  
La barra de progreso muestra el avance en tiempo real.


In [7]:
import numpy as np
from tqdm.auto import tqdm

textos  = df["texto_embedding"].tolist()
n       = len(textos)
batches = [textos[i:i+BATCH_SIZE] for i in range(0, n, BATCH_SIZE)]

print(f"Total registros : {n:,}")
print(f"Batches         : {len(batches)} × {BATCH_SIZE}")
print()

all_embeddings = []
t_start        = time.perf_counter()

for batch in tqdm(batches, desc="Vectorizando", unit="batch"):
    embs = model.encode(
        batch,
        normalize_embeddings=NORMALIZE,
        show_progress_bar=False,
    )
    all_embeddings.append(embs)

embeddings = np.vstack(all_embeddings).astype(np.float32)

t_end = time.perf_counter()
elapsed = t_end - t_start

print(f"\nShape embeddings : {embeddings.shape}")
print(f"Dtype            : {embeddings.dtype}")
print(f"Tiempo total     : {elapsed:.1f} s")
print(f"Tiempo / registro: {elapsed / n * 1000:.1f} ms")


Total registros : 4,070
Batches         : 64 × 64



Vectorizando: 100%|██████████| 64/64 [11:00<00:00, 10.33s/batch]


Shape embeddings : (4070, 1024)
Dtype            : float32
Tiempo total     : 660.9 s
Tiempo / registro: 162.4 ms


## 6 · Guardar en disco

- **`embeddings.npy`** — matriz `float32` de forma `(N, 1024)`.  
- **`embeddings_index.csv`** — metadatos de cada fila: `row_idx`, `id`, `title`,  
  `categoria`, `municipio`, `territorio`, `fuente`.


In [8]:
# ── Guardar vectores ─────────────────────────────────────────────────
np.save(EMB_PATH, embeddings)
print(f"Embeddings guardados en : {EMB_PATH}  ({EMB_PATH.stat().st_size / 1e6:.1f} MB)")

# ── Guardar índice de metadatos ───────────────────────────────────────
index_cols = ["id", "title", "categoria", "municipio", "territorio", "fuente"]
# Solo incluimos las columnas que existen en el dataframe
index_cols = [c for c in index_cols if c in df.columns]

df_index = df[index_cols].copy()
df_index.insert(0, "row_idx", df_index.index)  # fila en el .npy
df_index.to_csv(INDEX_PATH, index=False, encoding="utf-8")

print(f"Índice guardado en      : {INDEX_PATH}")
print(f"\nVista previa del índice:")
df_index.head(5)


Embeddings guardados en : npc\embeddings.npy  (16.7 MB)
Índice guardado en      : npc\embeddings_index.csv

Vista previa del índice:


,row_idx,id,title,categoria,municipio,territorio,fuente
0,0,1,Dendaraba,comercio,Vitoria-Gasteiz,araba,https://turismoa.euskadi.eus/es/centros-comerc...
1,1,2,El Boulevard,comercio,Vitoria-Gasteiz,araba,https://turismoa.euskadi.eus/es/centros-comerc...
2,2,3,Parque Comercial Gorbeia,comercio,Zigoitia,araba,https://turismoa.euskadi.eus/es/centros-comerc...
3,3,4,Lakua Centro,comercio,Vitoria-Gasteiz,araba,https://turismoa.euskadi.eus/es/centros-comerc...
4,4,5,Artea,comercio,Leioa,bizkaia,https://turismoa.euskadi.eus/es/centros-comerc...


## 7 · Verificación de integridad

Recargamos los ficheros y comprobamos que los embeddings son correctos  
haciendo una búsqueda de prueba por similitud coseno.


In [9]:
# ── Recargar desde disco ─────────────────────────────────────────────
embs_cargados = np.load(EMB_PATH)
idx_cargado   = pd.read_csv(INDEX_PATH)

assert embs_cargados.shape[0] == len(idx_cargado), "⚠ Mismatch filas vs índice!"
assert embs_cargados.shape[1] == 1024,              "⚠ Dimensión inesperada!"

print(f"✓ Embeddings recargados : {embs_cargados.shape}")
print(f"✓ Índice recargado      : {len(idx_cargado):,} filas")

# ── Búsqueda de prueba ────────────────────────────────────────────────
CONSULTA_TEST = "query: museos de arte en Bilbao"
TOP_K         = 5

emb_q  = model.encode(CONSULTA_TEST, normalize_embeddings=True)
scores = embs_cargados @ emb_q          # producto escalar = coseno (vectores normalizados)
top_k  = np.argsort(scores)[::-1][:TOP_K]

print(f"\nBúsqueda de prueba: '{CONSULTA_TEST}'")
print(f"{'Score':>6}  {'Título':<50}  {'Municipio':<20}  Categoría")
print("-" * 110)
for idx in top_k:
    row = idx_cargado.iloc[idx]
    titulo    = str(row.get("title",     "") or "")[:48]
    municipio = str(row.get("municipio", "") or "")[:18]
    categoria = str(row.get("categoria", "") or "")[:40]
    print(f"  {scores[idx]:.3f}  {titulo:<50}  {municipio:<20}  {categoria}")


✓ Embeddings recargados : (4070, 1024)
✓ Índice recargado      : 4,070 filas

Búsqueda de prueba: 'query: museos de arte en Bilbao'
 Score  Título                                              Municipio             Categoría
--------------------------------------------------------------------------------------------------------------
  0.872  Bilbao alternativo                                  Bilbao                plan
  0.870  Museo de reproducciones artísticas                  Bilbao                cultura
  0.865  Museo de Arte Sacro                                 Bilbao                cultura
  0.863  Galería J. M. Lumbreras                             Bilbao                cultura
  0.861  Museo de Pasos de Semana Santa                      Bilbao                cultura


## 8 · Resumen de ficheros generados

| Fichero | Descripción |
|---|---|
| `npc/embeddings.npy` | Matriz `float32` `(N, 1024)` con los vectores |
| `npc/embeddings_index.csv` | Metadatos de cada fila (id, título, categoría…) |

**Cómo cargarlos en otro notebook:**

```python
import numpy as np
import pandas as pd

embeddings = np.load("npc/embeddings.npy")          # shape (N, 1024)
index      = pd.read_csv("npc/embeddings_index.csv")

# Búsqueda por similitud coseno
scores = embeddings @ emb_query   # emb_query normalizado con prefijo query:
top_k  = scores.argsort()[::-1][:10]
print(index.iloc[top_k])
```
